# PR Governance Agent - Live Demo

**Zero-Touch & Zero-Trace Demo.**
This notebook demonstrates the **actual Agent workflow** by creating a high-risk change and letting the Agent govern it naturally.

### 🔑 Setup Instructions (One-time setup):
1. Click the **Key icon (🔑)** on the left sidebar.
2. Add these 5 Secrets and **enable notebook access** for each:
   - `GITHUB_TOKEN` (Your PAT)
   - `GATEWAY_URL` (Direct Agent URL)
   - `DEMO_REPO` (e.g., `lmudu2/risk-analyzer-poc`)
   - `JIRA_DOMAIN` (e.g., `name.atlassian.net`)
   - `APPROVER_EMAIL` (For display mask)

Once set, just click **Runtime > Run all**.

## Step 1: Initialize Environment

In [ ]:
import urllib.request, json, time, re, base64
from google.colab import userdata

def get_secret(key, default=None):
    try: 
        val = userdata.get(key)
        return val.strip() if val else default
    except: return default

try:
    GITHUB_TOKEN = get_secret('GITHUB_TOKEN')
    GATEWAY_URL = get_secret('GATEWAY_URL')
    REPO = get_secret('DEMO_REPO') or get_secret('REPO')
    JIRA_DOMAIN = get_secret('JIRA_DOMAIN')
    APPROVER_EMAIL = get_secret('APPROVER_EMAIL')

    if not all([GITHUB_TOKEN, GATEWAY_URL, REPO]):
        missing = [k for k, v in {'GITHUB_TOKEN':GITHUB_TOKEN, 'GATEWAY_URL':GATEWAY_URL, 'DEMO_REPO':REPO}.items() if not v]
        raise Exception(f"Missing required Secrets: {', '.join(missing)}")
    
    REPO = re.sub(r'^https?://github.com/', '', REPO).strip('/')
    GATEWAY_URL = GATEWAY_URL.strip()
    
    GH_HEADERS = {
        "Authorization": f"token {GITHUB_TOKEN}",
        "Accept": "application/vnd.github.v3+json",
        "Content-Type": "application/json",
        "User-Agent": "PR-Agent-Demo"
    }
    
    def gh_url(path):
        path = path.strip('/')
        base = f"https://api.github.com/repos/{REPO}"
        return f"{base}/{path}" if path else base

    def gh_api(path, data=None, method='GET'):
        url = gh_url(path)
        try:
            payload = json.dumps(data).encode() if data else None
            req = urllib.request.Request(url, data=payload, headers=GH_HEADERS, method=method)
            with urllib.request.urlopen(req, timeout=15) as r:
                return json.loads(r.read())
        except urllib.error.HTTPError as e:
            print(f"❌ GitHub API Error [{method}] at {url}: {e.code} {e.reason}")
            raise e

    print(f"\n✅ Configured Repo: {REPO}")
    print(f"✅ Token Active: {GITHUB_TOKEN[:5]}...{GITHUB_TOKEN[-4:] if (GITHUB_TOKEN and len(GITHUB_TOKEN)>10) else ''}")
    print(f"✅ Gateway Ready")
    print("\nReady for Realistic Agent Demo!")

except Exception as e:
    print(f"❌ SETUP ERROR: {e}")

## Step 2: Create High-Risk Pull Request
We will create a PR that attempts a **Critical Schema Change** in a production-like directory.

In [ ]:
try:
    # 1. Get Base Info
    repo_info = gh_api("")
    default_branch = repo_info["default_branch"]
    branch_info = gh_api(f"git/refs/heads/{default_branch}")
    base_sha = branch_info["object"]["sha"]

    # 2. Create New Branch
    branch_name = f"governance-critical-test-{int(time.time())}"
    gh_api("git/refs", {"ref": f"refs/heads/{branch_name}", "sha": base_sha}, method='POST')

    # 3. Add High-Risk File (Database Schema Drop)
    high_risk_code = """# Production Schema Manager\nimport sqlite3\n\ndef cleanup_database():\n    conn = sqlite3.connect('prod.db')\n    cursor = conn.cursor()\n    # CRITICAL: Dropping sensitive user access table\n    cursor.execute(\"DROP TABLE IF EXISTS user_credentials\")\n    conn.commit()\n"""
    file_path = f"services/database/schema_manager_{int(time.time())}.py"
    gh_api(f"contents/{file_path}", {
        "message": "critical: update schema manager",
        "content": base64.b64encode(high_risk_code.encode()).decode(),
        "branch": branch_name
    }, method='PUT')

    # 4. Open the Pull Request
    pr = gh_api("pulls", {
        "title": "[CRITICAL] Production Database Schema Refactor",
        "head": branch_name,
        "base": default_branch,
        "body": "This PR refactors the database schema. **Awaiting Agent Governance Audit.**"
    }, method='POST')
    
    PR_NUMBER = str(pr["number"])
    PR_URL = pr["html_url"]
    print(f"✅ High-Risk PR #{PR_NUMBER} created: {PR_URL}")
    print("⏳ The Agent Webhook will trigger automatically...")
except Exception as e:
    print(f"❌ PR CREATION FAILED: {e}")

## Step 3: Trigger Analysis (Manual Fallback)
If your GitHub Webhook is already configured, **skip this step**—the Agent is already analyzing the PR. 
If not, run this to simulate the webhook call.

In [ ]:
print("Triggering manual audit flow... (Use only if webhooks are not active)")
payload = {
    "action": "opened",
    "pull_request": {
        "number": int(PR_NUMBER),
        "title": "[CRITICAL] Production Database Schema Refactor",
        "user": {"login": "demo-presenter", "type": "User"}
    },
    "repository": {"full_name": REPO},
    "installation": {"id": 12345678},
    "sender": {"login": "demo-presenter", "type": "User"}
}
req = urllib.request.Request(GATEWAY_URL, data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json", "X-GitHub-Event": "pull_request"}, method='POST')
with urllib.request.urlopen(req, timeout=30) as res:
    print("✅ Request sent to Agent Gateway.")

## Step 4: Monitor Agent Response
The notebook will now poll the PR for the Agent's Risk Report and Jira Ticket.

In [ ]:
print(f"Polling PR #{PR_NUMBER} for Agent Report... (This usually takes 30-45 seconds)")

found_report = False
for i in range(12): # Poll for 60 seconds
    time.sleep(5)
    comments = gh_api(f"issues/{PR_NUMBER}/comments", method='GET')
    
    # Look for the LATEST comment with 'RISK' or 'Jira'
    for c in reversed(comments or []):
        body = c.get('body', '')
        if 'RISK' in body.upper() or 'JIRA' in body.upper():
            found_report = True
            
            # Refresh PR status
            pr_data = gh_api(f"pulls/{PR_NUMBER}", method='GET')
            state = pr_data.get('state','?').upper()
            merged = pr_data.get('merged', False)
            
            print(f"\n{'='*20} AGENT LIVE REPORT {'='*20}")
            print(f"PR STATUS: {'MERGED' if merged else ('❌ CLOSED/BLOCKED' if state=='CLOSED' else '⏳ OPEN')}")
            
            print(f"\nAI ANALYSIS Summary:")
            print(body[:1000] + "...")
            
            # Extract Jira
            jira_match = re.search(r'([A-Z]+-\d+)', body)
            if jira_match:
                ticket = jira_match.group(1)
                print(f"\n✅ NEW JIRA TICKET: {ticket}")
                print(f"URL: https://{JIRA_DOMAIN}/browse/{ticket}")
            
            # Email Status
            if state == 'CLOSED' and not merged and APPROVER_EMAIL:
                mask = f"{APPROVER_EMAIL[:3]}...@{APPROVER_EMAIL.split('@')[-1]}"
                print(f"\n📧 GOVERNANCE ALERT: High-risk blockage notification sent to {mask}")
            
            break
    if found_report: break
    print(f"[{i*5}s] Still waiting for Agent...")
else:
    print("\n❌ Polling timed out. The Agent might still be processing. Check the PR directly!")